**학습 목표**: 같은 데이터·같은 평가지표로 두 앙상블 모델을 비교하고, 성능과 특성중요도의 차이를 관찰할 수 있다.

In [2]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier


In [10]:
def compare_rf_xgb(X: np.ndarray, y: np.ndarray, random_state: int = 42) -> dict:
    """
    요구사항:
    - train_test_split으로 데이터를 나눈다.
    - RandomForestClassifier와 XGBClassifier(또는 xgboost 미설치 시 GradientBoostingClassifier)를
      각각 같은 train 세트로 학습한다.
    - 테스트 세트에서 accuracy와 ROC-AUC를 각각 계산한다.
    - 두 모델의 feature_importances_ 상위 5개를 각각 반환한다.
    - 결과를 dict로 반환 (모델명 → {accuracy, roc_auc, top5_features}).
    """
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
    randomforest_model = RandomForestClassifier()
    XGB_model = XGBClassifier() or GradientBoostingClassifier()
    models = []
    models.append(randomforest_model)
    models.append(XGB_model)
    results = {}
    for model in models:
        model.fit(X_train, y_train)
        model.predict_proba(X_test)[:,1]
        Accuracy = model.score(X_test, y_test)
        ROC_AUC = roc_auc_score(y_test, model.predict_proba(X_test), multi_class="ovr")
        idx = np.argsort(model.feature_importances_)[::-1]
        importances = model.feature_importances_[idx][:5]
        results[model] = {'accuracy' : Accuracy, 'roc_auc' : ROC_AUC, 'importances' : importances}
    
    return results



학습 목표: Day1~3에서 만든 compare_rf_xgb, best_split이 breast_cancer 말고 다른 데이터에도 수정 없이 통하는지 확인한다.
구현 과제 (스스로 작성): load_wine(다중 클래스, 3개)에 compare_rf_xgb를 적용해보세요. 이진분류를 가정하고 짠 부분(예: ROC-AUC 계산)이 있다면 다중 클래스에서 에러가 나거나 의미가 달라질 수 있습니다 — 그 지점을 스스로 찾아 수정하세요(roc_auc_score의 multi_class 옵션을 찾아보는 것도 방법입니다).
확인 질문: 이진분류용으로 짠 코드가 다중분류에서 어디서

In [11]:
from sklearn.datasets import load_wine

data = load_wine()
X, y = data.data, data.target
result = compare_rf_xgb(X, y)
for model_name, metrics in result.items():
    print(f"\n{model_name}")
    print(f"  accuracy: {metrics['accuracy']:.4f}")
    print(f"  roc_auc : {metrics['roc_auc']:.4f}")
    print(f"  top5   : {metrics['importances']}")


RandomForestClassifier()
  accuracy: 1.0000
  roc_auc : 1.0000
  top5   : [0.17122342 0.15054602 0.1495702  0.11991969 0.11802965]

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)
  accuracy: 0.9815
  roc_auc : 0.9986
  top5   : [0.3346779  0.17019057 0.16397238 0.14925942 0.07852591]
